In [ ]:
%pip install keras

In [3]:
import os
import pandas as pd
from datasets import load_dataset
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
print("📥 Loading datasets from Hugging Face Hub...")

dataset = load_dataset("guillherms/human-activity-pose_v4")

# Converter para pandas DataFrames
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()

print(f"✅ Train: {len(train_df)} rows | Validation: {len(val_df)} rows")
print("📊 Columns:", list(train_df.columns)[:10], "...")

📥 Loading datasets from Hugging Face Hub...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train/train.parquet:   0%|          | 0.00/795k [00:00<?, ?B/s]

validation/validation.parquet:   0%|          | 0.00/251k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/736 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/184 [00:00<?, ? examples/s]

✅ Train: 736 rows | Validation: 184 rows
📊 Columns: ['x0', 'y0', 'z0', 'v0', 'x1', 'y1', 'z1', 'v1', 'x2', 'y2'] ...


In [5]:
feature_cols = [c for c in train_df.columns if c not in ["label", "description"]]
X_train, y_train = train_df[feature_cols], train_df["label"]
X_val, y_val = val_df[feature_cols], val_df["label"]

# Codifica labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)
num_classes = len(le.classes_)

print(f"🧩 Classes: {list(le.classes_)}")

🧩 Classes: ['agreeing', 'dancing', 'handshake', 'lying_down', 'medical_observation', 'medical_procedure', 'office_work', 'reading', 'waving']


In [6]:
print("\n🌲 Training RandomForestClassifier...")
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train, y_train_enc)
rf_preds = rf.predict(X_val)
rf_acc = accuracy_score(y_val_enc, rf_preds)
print(f"✅ RF Accuracy: {rf_acc:.4f}")
print(classification_report(y_val_enc, rf_preds, target_names=le.classes_))
joblib.dump(rf, "rf_pose.pkl")


🌲 Training RandomForestClassifier...
✅ RF Accuracy: 0.9891
                     precision    recall  f1-score   support

           agreeing       1.00      1.00      1.00        12
            dancing       1.00      1.00      1.00        24
          handshake       1.00      1.00      1.00         7
         lying_down       1.00      1.00      1.00        30
medical_observation       1.00      0.94      0.97        16
  medical_procedure       1.00      1.00      1.00        31
        office_work       1.00      1.00      1.00        22
            reading       0.94      1.00      0.97        30
             waving       1.00      0.92      0.96        12

           accuracy                           0.99       184
          macro avg       0.99      0.98      0.99       184
       weighted avg       0.99      0.99      0.99       184



['rf_pose.pkl']

In [7]:
print("\n🤖 Training MLP Neural Network...")
mlp = Sequential([
    Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(num_classes, activation='softmax')
])
mlp.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
history = mlp.fit(X_train, y_train_enc, validation_data=(X_val, y_val_enc),
                  epochs=25, batch_size=64, callbacks=[es], verbose=1)

mlp_acc = max(history.history["val_accuracy"])
print(f"✅ MLP Accuracy: {mlp_acc:.4f}")
mlp.save("mlp_pose.h5")


🤖 Training MLP Neural Network...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 5s 121ms/step - accuracy: 0.3431 - loss: 1.8144 - val_accuracy: 0.7500 - val_loss: 1.0178
Epoch 2/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.7357 - loss: 0.9433 - val_accuracy: 0.8098 - val_loss: 0.5981
Epoch 3/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.8116 - loss: 0.6549 - val_accuracy: 0.9022 - val_loss: 0.3990
Epoch 4/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8578 - loss: 0.4356 - val_accuracy: 0.9565 - val_loss: 0.2649
Epoch 5/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.8896 - loss: 0.3643 - val_accuracy: 0.9674 - val_loss: 0.2014
Epoch 6/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9236 - loss: 0.3095 - val_accuracy: 0.9674 - val_loss: 0.1523
Epoch 7/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9403 - loss: 0.2291 - val_accuracy: 0.9565 - val_loss: 0.1376
Epoch 8/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9619 - loss: 0.1645 - val_accuracy: 0.9728 - 

✅ MLP Accuracy: 0.9946


In [9]:
print("\n⏳ Preparing sequences for LSTM (temporal model)...")

SEQ_LEN = 30

def create_sequences(X, y, seq_len):
    """
    Cria janelas de sequência temporal para treino LSTM.
    Aceita y como pandas.Series ou numpy.ndarray.
    """
    # Suporte para y em formato numpy
    if hasattr(y, "iloc"):
        get_label = lambda idx: y.iloc[idx]
    else:
        get_label = lambda idx: y[idx]

    sequences, labels = [], []
    for i in range(0, len(X) - seq_len, seq_len):
        seq = X.iloc[i:i+seq_len].values
        label = get_label(i + seq_len - 1)
        sequences.append(seq)
        labels.append(label)

    return np.array(sequences), np.array(labels)

# Gerar dados temporais
X_train_seq, y_train_seq = create_sequences(X_train, y_train_enc, SEQ_LEN)
X_val_seq, y_val_seq = create_sequences(X_val, y_val_enc, SEQ_LEN)

print(f"✅ LSTM sequences: {X_train_seq.shape} train | {X_val_seq.shape} val")

# Construir o modelo
model_lstm = Sequential([
    LSTM(128, input_shape=(SEQ_LEN, X_train_seq.shape[2])),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model_lstm.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

es_lstm = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

print("\n🎥 Training LSTM (temporal model)...")
hist_lstm = model_lstm.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=30, batch_size=32,
    callbacks=[es_lstm], verbose=1
)

# Avaliar
lstm_acc = max(hist_lstm.history["val_accuracy"])
print(f"\n✅ LSTM Accuracy: {lstm_acc:.4f}")

# Salvar modelo e encoder
model_lstm.save("lstm_pose.h5")
joblib.dump(le, "label_encoder.pkl")

print("\n💾 Model and encoder saved:")
print(" - lstm_pose.h5")
print(" - label_encoder.pkl")


⏳ Preparing sequences for LSTM (temporal model)...
✅ LSTM sequences: (24, 30, 132) train | (6, 30, 132) val


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



🎥 Training LSTM (temporal model)...
Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - accuracy: 0.0833 - loss: 2.2747 - val_accuracy: 0.6667 - val_loss: 2.0997
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.3333 - loss: 2.0492 - val_accuracy: 0.1667 - val_loss: 2.1265
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.2083 - loss: 2.0436 - val_accuracy: 0.1667 - val_loss: 2.1574
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.1250 - loss: 1.9713 - val_accuracy: 0.1667 - val_loss: 2.1592
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.2917 - loss: 1.9160 - val_accuracy: 0.5000 - val_loss: 2.1631
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.2083 - loss: 1.9463 - val_accuracy: 0.1667 - val_loss: 2.1714



✅ LSTM Accuracy: 0.6667

💾 Model and encoder saved:
 - lstm_pose.h5
 - label_encoder.pkl


In [10]:
print("\n📊 Final Results:")
print(f"RandomForest: {rf_acc:.4f}")
print(f"MLP: {mlp_acc:.4f}")
print(f"LSTM: {lstm_acc:.4f}")

best_model = max(
    [("rf_pose.pkl", rf_acc), ("mlp_pose.h5", mlp_acc), ("lstm_pose.h5", lstm_acc)],
    key=lambda x: x[1]
)

print(f"\n🏆 Best model: {best_model[0]} ({best_model[1]:.4f})")

# Salvar o encoder de classes
joblib.dump(le, "label_encoder.pkl")

print("\n✅ Training benchmark completed successfully!")


📊 Final Results:
RandomForest: 0.9891
MLP: 0.9946
LSTM: 0.6667

🏆 Best model: mlp_pose.h5 (0.9946)

✅ Training benchmark completed successfully!


In [ ]:
%pip install huggingface_hub

In [13]:
from huggingface_hub import HfApi,login,create_repo
from google.colab import userdata

huggingface_token = userdata.get('huggingface_token')
login(token=huggingface_token)

api = HfApi()
repo_id = "guillherms/human-activity-pose-models"

# Cria o repositório se ainda não existir
create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# Upload do modelo MLP
api.upload_file(
    path_or_fileobj="/content/mlp_pose.h5",
    path_in_repo="mlp_pose.h5",
    repo_id=repo_id,
    repo_type="model"
)

# Upload do LabelEncoder
api.upload_file(
    path_or_fileobj="/content/label_encoder.pkl",
    path_in_repo="label_encoder.pkl",
    repo_id=repo_id,
    repo_type="model"
)

readme = """---
language:
- en
license: mit
library_name: keras
tags:
- human-activity-recognition
- pose-estimation
- mediapipe
- tensorflow
datasets:
- guillherms/human-activity-pose_v4
pipeline_tag: image-classification
pretty_name: Human Activity Recognition MLP
---

# 🧠 Human Activity Recognition MLP

This model classifies human activities (reading, waving, office work, etc.)
using pose landmarks extracted from **MediaPipe Pose**.

## 📊 Architecture
- Multi-Layer Perceptron (Dense NN)
- Input: 132 pose landmark features
- Output: 10 activity classes

## 🧩 Files
- `mlp_pose.h5`: Trained Keras model
- `label_encoder.pkl`: Encodes activity labels

## 🧠 Author
Created by **Guilherme Santos**
"""

with open("README.md", "w") as f:
    f.write(readme)

api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id="guillherms/human-activity-pose-models",
    repo_type="model"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/mlp_pose.h5        : 100%|##########|  849kB /  849kB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/label_encoder.pkl  : 100%|##########|   594B /   594B            

CommitInfo(commit_url='https://huggingface.co/guillherms/human-activity-pose-models/commit/b98c66707687805af1dd77f91399f0b05d1ac688', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='b98c66707687805af1dd77f91399f0b05d1ac688', pr_url=None, repo_url=RepoUrl('https://huggingface.co/guillherms/human-activity-pose-models', endpoint='https://huggingface.co', repo_type='model', repo_id='guillherms/human-activity-pose-models'), pr_revision=None, pr_num=None)